In [15]:
# importing libraries

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score ,
    precision_score ,
    recall_score ,
    f1_score ,
    average_precision_score , 
    roc_auc_score ,
    confusion_matrix ,
    classification_report, 
    ConfusionMatrixDisplay 
)

In [16]:
catboost_oof = pd.read_csv("../outputs/oof/catboost_oof.csv")
lightgbm_oof = pd.read_csv("../outputs/oof/lightgbm_oof.csv")
xgboost_oof = pd.read_csv("../outputs/oof/xgboost_oof.csv")

In [18]:
print(catboost_oof.shape)
print(lightgbm_oof.shape)
print(xgboost_oof.shape)

print(np.array_equal(catboost_oof["isFraud"].values , lightgbm_oof["isFraud"].values))
print(np.array_equal(catboost_oof["isFraud"].values , xgboost_oof["isFraud"].values))

(590540, 2)
(590540, 2)
(590540, 2)
True
True


In [19]:
print(catboost_oof["catboost_oof"].isna().sum())
print(lightgbm_oof["lightgbm_oof"].isna().sum())
print(xgboost_oof["xgboost_oof"].isna().sum())

98425
98425
98425


In [20]:
ensemble_df = pd.DataFrame({
    "isFraud" : catboost_oof["isFraud"] , 
    "catboost" : catboost_oof["catboost_oof"] ,
    "lightgbm" : lightgbm_oof["lightgbm_oof"] ,
    "xgboost" : xgboost_oof["xgboost_oof"]
})

In [39]:
ensemble_df.tail(10)

,isFraud,catboost,lightgbm,xgboost,ensemble
590530,0,0.010519,0.009777,0.005424,0.008573
590531,0,0.029270,0.019447,0.006049,0.018256
590532,0,0.003257,0.006798,0.023041,0.011032
590533,0,0.006484,0.014330,0.063458,0.028090
590534,0,0.037834,0.031940,0.026051,0.031941
590535,0,0.006683,0.008476,0.004192,0.006450
590536,0,0.005940,0.008443,0.003722,0.006035
590537,0,0.005000,0.004684,0.003811,0.004498
590538,0,0.012628,0.064819,0.017558,0.031668
590539,0,0.030854,0.013721,0.006375,0.016984


In [21]:
ensemble_df["ensemble"] = (ensemble_df["catboost"] + ensemble_df["lightgbm"] + ensemble_df["xgboost"]) / 3 

In [22]:
mask = ensemble_df["ensemble"].notna()
y_oof = ensemble_df.loc[mask,"isFraud"]
ensemble_pred = ensemble_df.loc[mask , "ensemble"]

In [25]:
ensemble_roc_auc = roc_auc_score(y_oof , ensemble_pred)
ensemble_pr_auc = average_precision_score(y_oof , ensemble_pred)

print(f"Ensemble OOF ROC-AUC : {ensemble_roc_auc:.4f}")
print(f"Ensemble OOF PR-AUC : {ensemble_pr_auc:.4f}")

Ensemble OOF ROC-AUC : 0.9208
Ensemble OOF PR-AUC : 0.6127


In [38]:
ensemble_class = (ensemble_pred >= 0.5).astype(int)
print("Confusion Matrix: \n")
print(confusion_matrix(y_oof , ensemble_class))

print(f"Precision : {precision_score(y_oof , ensemble_class):.4f}")
print(f"Recall : {recall_score(y_oof , ensemble_class):.4f}")
print(f"F1 Score : {f1_score(y_oof , ensemble_class):.4f}")

Confusion Matrix: 

[[473108    873]
 [ 11587   6547]]
Precision : 0.8823
Recall : 0.3610
F1 Score : 0.5124
